In [50]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
df = pd.read_csv('customer_shopping_behavior.csv')

df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [11]:
df.describe(include='all')     # select all columns

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [13]:
# check missing values
df.isnull().sum()

# missing values present on Review Rating in 37 rows

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [17]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [22]:
# Edit columns name with lowercase and replace spaces to '_' for use columns name in code withous errors

df.columns = df.columns.str.lower()           # convert all column with lowercase
df.columns = df.columns.str.replace(' ','_')   # replace 'spalce' between words to '_'

df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [24]:
# Create columns Age_groups
label = ['Young Adult', 'Adult', 'Middel Aged', 'Senior']
df['age_group'] = pd.qcut(x=df['age'], q=4, labels=label)

# qcut --> Quantile-based discretization function

In [30]:
# Create a newcolumns - purchase_frequency_days
frequency_days = { 'Fortnightly': 14, 
                   'Weekly': 7, 
                   'Monthly': 30,
                   'Every 3 Months': 90,
                   'Annually':365, 
                   'Quarterly':90, 
                   'Bi-Weekly':14,}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_days)

In [41]:
# both row data's are similar/identical
df[['discount_applied','promo_code_used']]

# check both rows data is similar
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [46]:
# Remove column 'promo_code_used'
df = df.drop('promo_code_used', axis=1)

## create connection and load data into mysql

In [58]:
from sqlalchemy import create_engine, text
import pandas as pd

# 1. Define your connection credentials
USER = 'root'
PASSWORD = 'rutvikkihla'
HOST = 'localhost'       
PORT = '3306'            
DATABASE = 'customer_behavior'

# 2. Connect to the base MySQL server first (without specifying a database)
base_engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}")

# 3. Create the database automatically if it doesn't exist
with base_engine.connect() as conn:
    # We use backticks `` ` `` because your database name contains a space
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{DATABASE}`;"))
    print(f"Database '{DATABASE}' is ready!")

# 4. Now create the targeted engine pointing directly to your database
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

# 5. Load the DataFrame into the 'customer' table
df.to_sql(
    name='customer',      
    con=engine,           
    if_exists='replace',   
    index=False           
)

print("DataFrame successfully loaded into MySQL!")

Database 'customer_behavior' is ready!
DataFrame successfully loaded into MySQL!
